# S10, What a GPU buys a physics simulator


Measures what a GPU actually buys a physics simulator, and what it costs in
fidelity. **No training**, this finishes inside one free session and produces
numbers whether or not the quota holds out.

Three questions:

1. **Does MJX agree with MuJoCo C?** Not "do the trajectories overlap" (they
   never do, the systems are chaotic) but "does MJX diverge by more than
   float32 alone would".
2. **Steps per second against batch size**, with a threaded MuJoCo C baseline
   on the same model, so the speedup has a denominator.
3. **What the solver iteration budget costs and buys**, on the GPU, where the
   trade is different from CPU.

Run this first. It also tells you the batch size to configure S2 with.


---

### Before you run anything

1. **Runtime → Change runtime type → T4 GPU.** Every cell below assumes it.
2. **Keep this tab visible.** Free Colab disconnects an idle notebook after
   about 90 minutes and reclaims the runtime; `/content` does not survive it.
3. **The free tier has a quota you cannot see.** It is not published, it
   varies, and it is consumed by wall-clock GPU time whether or not you are
   computing. Expect a few hours a day, and expect to be cut off mid-run
   without warning. Every long-running cell here is written to survive that.

Checkpoints go to Google Drive, not to `/content`. That is the whole reason
the Drive cell exists, a run that checkpoints only to local disk loses
everything the moment the runtime is reclaimed, which on free Colab is the
normal way a session ends rather than an exceptional one.

## 1. Hardware, packages, Drive, code

In [ ]:
# --- what hardware did we actually get? ---
import subprocess, sys
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                      '--format=csv'], capture_output=True, text=True).stdout)

# A label is not hardware. A Kaggle session advertised as "T4 x2" reported a
# P100 to the driver, which has compute capability 6.0 and cannot run several
# things a T4 can. Record what the driver says and quote it as such; never
# write "measured on a T4" because the runtime menu said T4.

In [ ]:
# --- packages ---
# jax with CUDA is preinstalled on Colab GPU runtimes. mujoco-mjx is not, and
# installing it can drag in a CPU-only jax wheel that silently replaces the
# working one. So: install, then re-check the device, and only reinstall jax
# if the check fails.
import subprocess, sys

def sh(cmd):
    print('$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True)
    if r.returncode:
        raise SystemExit(f'command failed: {cmd}')

sh(f'{sys.executable} -m pip install -q mujoco mujoco-mjx optax')

import importlib, jax
importlib.reload(jax)
if not any(d.platform == 'gpu' for d in jax.devices()):
    print('jax lost the GPU during install; reinstalling the CUDA wheel')
    sh(f'{sys.executable} -m pip install -q -U "jax[cuda12]"')
    raise SystemExit(
        'Reinstalled jax. Runtime -> Restart session, then run this cell '
        'again. (A restart is required: the CPU-only jax is already imported '
        'into this process and reimporting will not replace it.)')

In [ ]:
import jax, mujoco
print('jax     ', jax.__version__)
print('mujoco  ', mujoco.__version__)
print('devices ', jax.devices())
print('kind    ', getattr(jax.devices()[0], 'device_kind', '?'), '(as reported by the driver)')

# Hard stop, not a warning. On CPU a single mjx.step of the LEAP scene costs
# about 4 seconds and the compile runs past half an hour: a CPU session is not
# a slow run, it is no run.
assert any(d.platform == 'gpu' for d in jax.devices()), \
    'No GPU. Runtime -> Change runtime type -> T4 GPU, then restart and re-run.'
print()
print('GPU OK')

In [ ]:
# --- Drive, for anything that must outlive this runtime ---
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive/robotics-rl-portfolio')
DRIVE.mkdir(parents=True, exist_ok=True)
WORK = Path('/content/work'); WORK.mkdir(exist_ok=True)
print('drive :', DRIVE)
print('local :', WORK)

In [ ]:
# --- code ---
import os, shutil, subprocess, sys
from pathlib import Path

SRC = WORK / 'robotics-rl-portfolio'
REPO = 'https://github.com/JacobEGarcia/robotics-rl-portfolio.git'

def sh(cmd, **kw):
    print('$', cmd, flush=True)
    return subprocess.run(cmd, shell=True, check=True, **kw)

if SRC.exists():
    sh(f'cd {SRC} && git pull --ff-only')
else:
    sh(f'git clone --depth 1 {REPO} {SRC}')

need = SRC / 's10_gpu_scaling/run.py'
if not need.exists():
    # Fallback for code that is committed locally but not pushed. Tar the
    # repo on your machine, drop it in Drive, and this picks it up:
    #   tar czf portfolio.tgz --exclude=assets --exclude=runs .
    tgz = DRIVE / 'portfolio.tgz'
    if tgz.exists():
        print(f'{need} missing from the clone; unpacking {tgz} over it')
        sh(f'tar xzf {tgz} -C {SRC}')
    if not need.exists():
        raise SystemExit(
            f'{need} is not in the cloned repo and no {tgz} was found.\n'
            f'Push it from your machine:\n'
            f'    cd ~/Downloads/hermestes/robotics-rl-portfolio && git push origin main\n'
            f'or upload a tarball to {tgz}.')

os.environ['PYTHONPATH'] = str(SRC)
sys.path.insert(0, str(SRC))
print()
print('code OK at', SRC)

In [ ]:
# --- the robot models ---
# assets/menagerie is gitignored in the portfolio repo on purpose: Menagerie
# is 2.3 GB of third-party assets and is itself a git repo, so it is fetched
# rather than vendored. A sparse checkout gets what is needed in a few
# seconds instead of pulling all of it.
MENAGERIE = SRC / 'assets' / 'menagerie'
WANT = ['leap_hand', 'franka_emika_panda', 'unitree_z1',
        'unitree_go1', 'unitree_h1', 'shadow_hand', 'dynamixel_2r']

if not (MENAGERIE / 'leap_hand' / 'right_hand.xml').exists():
    shutil.rmtree(MENAGERIE, ignore_errors=True)
    MENAGERIE.parent.mkdir(parents=True, exist_ok=True)
    sh('git clone --depth 1 --filter=blob:none --sparse '
       'https://github.com/google-deepmind/mujoco_menagerie.git ' + str(MENAGERIE))
    sh(f'cd {MENAGERIE} && git sparse-checkout set ' + ' '.join(WANT))

missing = [w for w in WANT if not (MENAGERIE / w).exists()]
assert not missing, f'sparse checkout did not produce: {missing}'
print('models OK:', sorted(p.name for p in MENAGERIE.iterdir() if p.is_dir())[:12])

## 2. Fidelity: MJX against MuJoCo C

About 10-20 minutes. The float64 pass runs as a subprocess because `jax_enable_x64` is process-global and MJX fixes its array dtypes when the model is built.

In [ ]:
import subprocess, sys, os
os.chdir(SRC)

MODELS = ['dynamixel_2r', 'unitree_z1', 'franka_emika_panda',
          'leap_hand', 's2_inhand', 'unitree_go1', 'unitree_h1']

for cmd in (['parity', '--steps', '300'],
            ['parity-x64', '--steps', '300'],
            ['integrator', '--steps', '40']):
    r = subprocess.run([sys.executable, '-u', '-m', 's10_gpu_scaling.run',
                        *cmd, '--models', *MODELS],
                       env=dict(os.environ, PYTHONPATH=str(SRC)))
    print('exit', r.returncode)

### What the numbers mean

For each model you get five curves. The two that decide the verdict:

* **floor B (`roundoff`)**, MuJoCo C, float64 arithmetic, state
  truncated to float32 after every step. A lower bound on what
  float32 costs, measured in the same binary with no implementation
  difference in it.
* **`mjx_f32`**, MJX against MuJoCo C at the same timestep.

`mjx_f32` near or below floor B means precision explains the gap.
`mjx_f64` still large is the only evidence of an algorithmic
difference, and then the place to look is the constraint solver.

Do **not** read `mjx_f32` against floor A (`envelope`, a single
float32-sized perturbation then pure float64). Floor A runs tens of
thousands of times below floor B and makes every port look broken.

In [ ]:
import json
from pathlib import Path
d = json.loads((SRC / 's10_gpu_scaling/results/parity.json').read_text())
print(f"{'model':<20}{'floorA':>11}{'floorB':>11}{'mjx_f32':>11}{'mjx_f64':>11}{'dt cost':>11}")
for k, v in d.items():
    if 'summary' not in v:
        print(f'{k:<20} {v.get("error","?")[:60]}'); continue
    g = lambda n: v['summary'].get(n, {}).get('final_m')
    fmt = lambda x: f'{x:>11.2e}' if x is not None else f'{"-":>11}'
    print(f'{k:<20}' + ''.join(fmt(g(n)) for n in
          ('envelope', 'roundoff', 'mjx_f32', 'mjx_f64', 'mjc_dt')))

### Localising a disagreement: the integrator

Measured on CPU, this is where the Panda's whole MJX/MuJoCo gap lives, and it
is neither precision nor contacts. Under `mjINT_EULER` the two agree to
1.4e-07 m, five orders of magnitude below the float32 floor. Under
`mjINT_IMPLICITFAST`, which is what `panda.xml`, `leap_hand` and **the S2
scene itself** ship, they are 16 mm apart within 40 ms. MJX refuses
`mjINT_IMPLICIT` outright.

The Euler row is also the positive control this study needs: it proves the
harness can show agreement, so the large numbers elsewhere are results rather
than bugs in the comparison. Check whether the CUDA backend reproduces it.

In [ ]:
d = json.loads((SRC / 's10_gpu_scaling/results/integrator.json').read_text())
for k, v in d.items():
    print(k, '(ships ' + v['shipped'] + ')')
    for r in v['rows']:
        if 'unsupported' in r:
            print(f"  {r['integrator']:<20} NOT SUPPORTED BY MJX")
        elif 'error' in r:
            print(f"  {r['integrator']:<20} {r['error'][:70]}")
        else:
            print(f"  {r['integrator']:<20} {r['mjx_vs_mjc_m']:.3e} m  "
                  f"floor {r['float32_floor_m']:.3e}  ratio {r['ratio_to_floor']:>9.5f}")

## 3. Throughput: steps/s against batch size

20-40 minutes. The sweep climbs until two consecutive allocations fail, so it finds the memory ceiling rather than assuming one. Compile time is reported separately: it is a real cost when you are sizing a job against a session that might last two hours.

In [ ]:
r = subprocess.run([sys.executable, '-u', '-m', 's10_gpu_scaling.run',
                    'throughput', '--max-envs', '16384',
                    '--models', 'dynamixel_2r', 'franka_emika_panda',
                    'leap_hand', 's2_inhand', 'unitree_go1'],
                   env=dict(os.environ, PYTHONPATH=str(SRC)))
print('exit', r.returncode)

In [ ]:
d = json.loads((SRC / 's10_gpu_scaling/results/throughput.json').read_text())
print(f"{'model':<20}{'cpu':>12}{'gpu best':>12}{'at n':>8}{'x cpu':>8}{'knee n':>8}{'compile':>9}")
for k, v in d.items():
    b, kn, c = v.get('best'), v.get('knee'), v.get('cpu_baseline', {})
    if not b:
        continue
    print(f"{k:<20}{c.get('steps_per_s',0):>12,.0f}{b['steps_per_s']:>12,.0f}"
          f"{b['n_envs']:>8,}{(v.get('gpu_over_cpu') or 0):>8.1f}"
          f"{(kn or {}).get('n_envs',0):>8,}{b['compile_s']:>8.0f}s")

s2 = d.get('s2_inhand', {})
if s2.get('knee'):
    print()
    print(f"=> configure S2 with --num-envs {s2['knee']['n_envs']}")
    print(f"   ({s2['knee']['steps_per_s']:,.0f} steps/s; "
          f"1e8 steps ~ {1e8/s2['knee']['steps_per_s']/3600:.1f} GPU-hours)")

## 4. The solver iteration budget

The knob that actually sets the speed/fidelity trade on a GPU. On CPU an environment whose constraints are already satisfied exits the solver early; under `vmap` the whole batch runs the full budget every step, so a value tuned on CPU is usually wrong here.

Worth noticing in the output: DeepMind's own MJX demo scene (`panda_mjx_cube`) ships `iterations=5` where `panda.xml` ships 100.

In [ ]:
r = subprocess.run([sys.executable, '-u', '-m', 's10_gpu_scaling.run',
                    'solver', '--n-envs', '512',
                    '--models', 'franka_emika_panda', 'leap_hand', 'unitree_go1'],
                   env=dict(os.environ, PYTHONPATH=str(SRC)))
print('exit', r.returncode)

## 5. Figures, and get the results off this machine

Copy to Drive **before** doing anything else. A runtime reclaimed with results only in `/content` produced nothing.

In [ ]:
r = subprocess.run([sys.executable, '-u', '-m', 's10_gpu_scaling.figures', '--dark'],
                   env=dict(os.environ, PYTHONPATH=str(SRC)))

import shutil
out = DRIVE / 's10_results'
out.mkdir(parents=True, exist_ok=True)
for sub in ('s10_gpu_scaling/results', 'media/s10'):
    src = SRC / sub
    if src.exists():
        shutil.copytree(src, out / Path(sub).name, dirs_exist_ok=True)
print('copied to', out)
print(sorted(p.name for p in out.rglob('*') if p.is_file()))